# Naive Bayes Classifier

## What Is Naive Bayes?

Naive Bayes is a **probabilistic classification algorithm** built on Bayes' Theorem. Rather than learning a decision boundary or measuring distances, it takes a fundamentally different approach — it asks:

> *Given these features, what is the probability that this point belongs to each class?*

It then predicts whichever class has the highest probability. This makes it a **generative model** — it models the underlying probability distribution of the data rather than just the boundary between classes.

## Bayes' Theorem

The foundation of the algorithm is Bayes' Theorem:

$$P(C \mid \mathbf{X}) = \frac{P(\mathbf{X} \mid C) \cdot P(C)}{P(\mathbf{X})}$$

Each term has a specific name and meaning:

| Term | Name | Meaning |
|---|---|---|
| $P(C \mid \mathbf{X})$ | **Posterior** | Probability of class $C$ given the observed features — what we want |
| $P(\mathbf{X} \mid C)$ | **Likelihood** | Probability of observing these features if the class were $C$ |
| $P(C)$ | **Prior** | How common class $C$ is in the training data, before seeing any features |
| $P(\mathbf{X})$ | **Evidence** | Overall probability of observing these features across all classes |

In words:

$$\text{Posterior} = \frac{\text{Likelihood} \times \text{Prior}}{\text{Evidence}}$$

Since $P(\mathbf{X})$ is the same for all classes, it does not affect which class wins. We can ignore it during classification and simply maximize the numerator.

## The "Naive" Assumption

The algorithm is called *naive* because of one strong simplifying assumption:

> **Features are conditionally independent given the class.**

This means:

$$P(X_1, X_2, \ldots, X_n \mid C) = P(X_1 \mid C) \cdot P(X_2 \mid C) \cdots P(X_n \mid C)$$

Instead of modeling all possible interactions between features — which becomes computationally intractable in high dimensions — we treat each feature as contributing independently to the probability. This reduces a massive joint probability into a simple product of individual probabilities.

Is this assumption realistic? Usually not. Features in real data are often correlated. But despite this, Naive Bayes works surprisingly well in practice, especially for text classification. The ranking of classes (which one is most probable) is often correct even when the exact probabilities are not.

## The Classification Rule

Combining Bayes' Theorem with the naive independence assumption, the final prediction rule is:

$$\hat{C} = \underset{C}{\arg\max} \ P(C) \prod_{i=1}^{n} P(X_i \mid C)$$

Where:

- $\hat{C}$ is the predicted class
- $\underset{C}{\arg\max}$ means "find the class $C$ that maximizes the expression"
- $P(C)$ is the prior probability of class $C$
- $\prod_{i=1}^{n} P(X_i \mid C)$ is the product of individual feature likelihoods

In practice, because multiplying many small probabilities together can cause **numerical underflow**, we take the logarithm and work with a sum instead:

$$\hat{C} = \underset{C}{\arg\max} \left[ \log P(C) + \sum_{i=1}^{n} \log P(X_i \mid C) \right]$$

This is mathematically equivalent (since $\log$ is monotonically increasing) and numerically stable.

## Gaussian Naive Bayes

The algorithm needs to estimate $P(X_i \mid C)$ — the likelihood of each feature given a class. The method of estimation depends on the type of data.

For **continuous features**, we assume each feature follows a **Gaussian (Normal) distribution** within each class:

$$P(x_i \mid C) = \frac{1}{\sqrt{2\pi\sigma_C^2}} \exp\left(-\frac{(x_i - \mu_C)^2}{2\sigma_C^2}\right)$$

Where:

- $\mu_C$ is the **mean** of feature $x_i$ for class $C$
- $\sigma_C^2$ is the **variance** of feature $x_i$ for class $C$
- $\pi \approx 3.14159$

During training, the model simply computes the mean and variance of each feature separately for each class, along with the prior $P(C)$ from class frequencies. There is no gradient descent, no iterative optimization, and no matrix inversion — it is just computing means and variances from data, which is extremely fast.

## Types of Naive Bayes

The way $P(X_i \mid C)$ is estimated varies depending on the nature of the features:

| Variant | Feature Type | How Likelihood Is Estimated |
|---|---|---|
| **Gaussian NB** | Continuous | Assumes Normal distribution; fits mean and variance per class |
| **Multinomial NB** | Count data (e.g. word counts) | Models feature counts; common in text classification |
| **Bernoulli NB** | Binary features | Models presence or absence of each feature |

For most tabular data with continuous features, Gaussian NB is the default choice.

## 1. Gaussian Naive Bayes

### Assumption

Each feature follows a **Normal (Gaussian) distribution** within each class:

$$P(x_i \mid C) = \frac{1}{\sqrt{2\pi\sigma_C^2}} \exp\left(-\frac{(x_i - \mu_C)^2}{2\sigma_C^2}\right)$$

Where:

- $x_i$ is the feature value for a given input
- $\mu_C$ is the **mean** of feature $x_i$ across all training examples of class $C$
- $\sigma_C^2$ is the **variance** of feature $x_i$ for class $C$

During training, the model simply computes $\mu_C$ and $\sigma_C^2$ for every feature–class combination. At prediction time, it plugs the new feature value into this formula to compute the likelihood.

### When To Use

Gaussian NB is appropriate when features are **continuous and roughly bell-shaped**. It is not required that the data be perfectly Gaussian — the model is robust to mild deviations.

Good use cases include sensor readings, physical measurements (height, weight), exam scores, and most tabular numeric data.

## 2. Multinomial Naive Bayes

### Assumption

Features represent **counts or frequencies** — typically word counts in a document. The model is based on the multinomial distribution:

$$P(C \mid d) \propto P(C) \prod_{i} P(w_i \mid C)^{f_i}$$

Where:

- $d$ is the document
- $w_i$ is a word in the vocabulary
- $f_i$ is the **frequency** (count) of word $w_i$ in the document
- $P(w_i \mid C)$ is the probability of word $w_i$ appearing in a document of class $C$

The model estimates $P(w_i \mid C)$ from training data as the relative frequency of each word within each class. A word that appears often in spam emails gets a high $P(w_i \mid \text{spam})$.

### When To Use

Multinomial NB is the standard choice for **text classification**. It works naturally with raw word counts and also with TF-IDF weighted features. The key requirement is that features are **non-negative discrete counts**.

Good use cases include spam detection, topic classification, and sentiment analysis.

## 3. Bernoulli Naive Bayes

### Assumption

Features are **binary** — each feature is either present (1) or absent (0). The likelihood is modeled as:

$$P(x_i \mid C) = \begin{cases} p_i & \text{if } x_i = 1 \\ 1 - p_i & \text{if } x_i = 0 \end{cases}$$

Where $p_i = P(x_i = 1 \mid C)$ is the probability that feature $i$ is present given class $C$.

The critical difference from Multinomial NB is that Bernoulli NB **explicitly penalizes the absence of a word**. If a word is absent from the document, the term $(1 - p_i)$ is multiplied in. Multinomial NB simply ignores absent words. This makes Bernoulli NB more sensitive to what is *not* in the document.

### When To Use

Bernoulli NB is appropriate when features are naturally binary — word presence vs. absence, yes/no flags, or any feature that is either on or off.

Good use cases include short-text classification, document presence/absence features, and any dataset where frequency does not matter but occurrence does.

## Summary Comparison

| Variant | Feature Type | What It Models | Typical Use Case |
|---|---|---|---|
| **Gaussian NB** | Continuous | Normal distribution per class | Sensor data, measurements |
| **Multinomial NB** | Count / frequency | Word count probabilities | Spam detection, topic classification |
| **Bernoulli NB** | Binary (0 or 1) | Presence or absence of features | Short text, binary feature sets |

The choice between Multinomial and Bernoulli NB is especially important in text tasks. If word **frequency** matters, use Multinomial. If only word **presence** matters, use Bernoulli.

## Classification Metrics

Since Naive Bayes is a classifier, we must use **classification metrics** — not $R^2$, which is for regression. The standard metrics are:

**Accuracy** — fraction of all predictions that were correct:

$$\text{Accuracy} = \frac{\text{Number of correct predictions}}{\text{Total predictions}}$$

**Precision** — of all points predicted as positive, how many actually were:

$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall** — of all actual positives, how many did we catch:

$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1 Score** — harmonic mean of Precision and Recall, useful when classes are imbalanced:

$$F_1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

Where $TP$ = True Positives, $FP$ = False Positives, $FN$ = False Negatives.

## When To Use Naive Bayes

**Use Naive Bayes when:**

- You need a fast, lightweight baseline classifier
- Working with text or high-dimensional data (spam detection, sentiment analysis)
- The dataset is small — Naive Bayes needs very little data to estimate its parameters
- Real-time prediction is needed — training and inference are both extremely fast

**Avoid Naive Bayes when:**

- Features are strongly correlated — the independence assumption is severely violated
- You need well-calibrated probability estimates — the probabilities output by Naive Bayes are often overconfident due to the independence assumption

## Comparison: Three Classifiers

| Model | What It Learns | Core Assumption |
|---|---|---|
| **Logistic Regression** | A linear decision boundary | Classes are linearly separable |
| **KNN** | Nothing — stores all training data | Nearby points share the same label |
| **Naive Bayes** | Probability distributions per class | Features are conditionally independent |

Each model makes a different bet about the structure of the data. Naive Bayes wins when data is high-dimensional and sparse (like text). KNN wins when the boundary is complex and the dataset is small. Logistic Regression wins when the boundary is approximately linear and interpretability matters.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# Synthetic dataset
texts = [
    "Win money now",
    "Limited offer claim prize",
    "Congratulations you won lottery",
    "Call now for free reward",
    "Meeting at 10 am tomorrow",
    "Project discussion schedule",
    "Let's have lunch today",
    "Submit the assignment by evening"
]

# Labels: 1 = Spam, 0 = Ham (Normal message)
labels = [1,1,1,1,0,0,0,0]

### Convert Text → Numerical Features (TF-IDF)

In [3]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(texts)
y = np.array(labels)

#### TF-IDF (Term Frequency – Inverse Document Frequency)

TF-IDF is a numerical statistic that reflects how **important a word is to a document within a collection**. Common words like "the" or "is" appear everywhere and carry little meaning — TF-IDF down-weights them automatically. Rare but meaningful words get higher scores.

It is computed as a product of two terms:

#### Term Frequency (TF)

How often a word appears in a document, normalized by document length:

$$TF = \frac{\text{word frequency in document}}{\text{total words in document}}$$

#### Inverse Document Frequency (IDF)

How rare the word is across the entire collection of documents:

$$IDF = \log\left(\frac{\text{total documents}}{\text{documents containing word}}\right)$$

If a word appears in every document, the fraction equals 1 and $\log(1) = 0$ — the word is given zero weight. The rarer the word across documents, the larger the IDF.

#### Final Score

$$TF\text{-}IDF = TF \times IDF$$

A word gets a high TF-IDF score only if it appears **frequently in this specific document** (high TF) but **rarely across other documents** (high IDF). This combination identifies words that are distinctive and informative for a particular document.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

model = MultinomialNB()

model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [6]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.5

Confusion Matrix:
 [[0 1]
 [0 1]]

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



In [8]:
new_text = ["Free lottery reward waiting"]

new_vector = vectorizer.transform(new_text)

prediction = model.predict(new_vector)

print("Prediction:", "Spam" if prediction[0] == 1 else "Ham")

Prediction: Spam
